# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks

## Part 1: Bloom Detection Method Testing

### Threshold Method

#### Function for determining the climatological threshold value
Based on the climatological median, this function finds a certain percentage of that median and adds it to the median to get a threshold value for determining phytoplankton blooms
* Must include:
    * Threshold percentage = thld (thld=0.05 by default)
    * Path to climatology files = path (set to NES Annual Climatology 1997 - 2020 by default)

In [ ]:
#Define a function to return the threshold value based on the regional climatology.
def threshold_value(thld = 0.05, path = None):
    if path == None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return(thld_value)

#### Plotting the Climatological Threshold

In [ ]:
clim_med = threshold_value().squeeze()
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
fig = plt.figure(figsize=(15,8))
map_projection = cartopy.crs.PlateCarree()
ax = plt.axes(projection=map_projection)
im = plt.pcolormesh(clim_med.lon,
                    clim_med.lat,
                    clim_med,
                    cmap=cmocean.cm.algae,
                    norm=LogNorm(vmin=0.1, vmax=10.0)
)
custom_ticks = [0.1,1,10]
cb = plt.colorbar(im,shrink=0.8,label='Chlorophyll a Threshold ($mg/m^3$)',ticks=custom_ticks,format='%g') #$ $ makes it a LaTEX function so it actually formats as an equation

ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree()) #Adding the shelf break line
ax.set_extent([-77,-62,37,47])
ax.gridlines(draw_labels=True)
ax.set_title('Chlorophyll a Climatological Median Threshold', fontsize=24)

#### Create a mask to filter data for bloom conditions

This function (bloom_mask_Boolean) takes the input data and determines if values exceed the climatological threshold determined above (or found via the threshold_value function). If the value is less than or equal to the threshold, then it is reported as false. If it is greater than the threshold, it is reported as true. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include
    * Path to files = path (set to D8 NES shelf files by default)

In [ ]:
def bloom_mask_Boolean(path=None): #Produces True and False values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        ds = xr.open_mfdataset(path)
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    return is_bloom_CHL

This function (bloom_mask_numeric) functions similar to bloom_mask_Boolean, except it reports false values as 0 and true values retain their actual value. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include:
    * Path to files = path (set to D8 NES shelf files by default)

In [ ]:
def bloom_mask_numeric(path=None): #Produces 0 and actual values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path,consolidated=True) #Opens zarr file, use until mfdatasets is working properly
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    clim_med_new = clim_med.squeeze('time', drop=True) #Removes time dimension from climatological mean, FIX THIS LINE
    is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

#### Mask Intervals
Finding the climatological threshold at a few different inteverals (5%, 10%, 15%, 20%, 25%, and 30%)

In [ ]:
thld_value = [0.05,0.1,0.15,0.2,0.25,0.3]
clim_med = threshold_value()
bloom_5 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[1])
bloom_10 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[2])
bloom_15 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[3])
bloom_20 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[4])
bloom_25 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[5])
bloom_30 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')

Plotting all of the masks (5% - 30%)
<br> Must pick a day of the year from (INPUT VALUE RANGE HERE)

In [ ]:
DOY = 7181
datasets = [
    (bloom_5[DOY],"Chlorophyll a 5% Mask"),
    (bloom_10[DOY], "Chlorophyll a 10% Mask"),
    (bloom_15[DOY], "Chlorophyll a 15% Mask"),
    (bloom_20[DOY], "Chlorphyll a 20% Mask"),
    (bloom_25[DOY], "Chlorophyll a 25% Mask"),
    (bloom_30[DOY], "Chlorophyll a 30% Mask"),
    ]

fig, axes = plt.subplots(3,2,figsize=(14,12),subplot_kw={"projection":map_projection})
axes_flat = axes.flatten()

im= None

for i, (data,title) in enumerate(datasets):
    ax = axes_flat[i]
    im = ax.pcolormesh(data.lon,
                data.lat,
                data,
                cmap=cmocean.cm.algae,
                norm=LogNorm(vmin=0.1, vmax=10.0)
    )
    ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
    ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
    ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree())
    ax.set_xlabel('Longitude ($^o$)', fontsize=12)
    ax.set_ylabel('Latitude ($^o$)', fontsize=12)
    ax.set_extent([-77,-63,34.5,46])
    ax.set_title(title, fontsize=14) #Plot headings
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False

custom_ticks = [0.1,1,10]
cb = fig.colorbar(im,ax=axes,shrink=0.5,label='Chlorophyll a Concentration ($mg/m^3$)',ticks=custom_ticks,format='%g')
fig.suptitle("Chlorophyll a Masks Based on NES Annual Climatology",fontsize=20) #Overall figure heading

#### Histograms of Data

Create the functions to subset the data

In [ ]:
ds = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)

Function to clip the median chl-a data for a specific longitude and latitude
* Must input 
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

In [ ]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None,region_title=None):
    if path == None:
        #ds = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        ds = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        ds = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    ds_local = ds.CHL_median.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    ds_local = ds_local.mean(dim=['lat','lon'])
    return ds_local

Function to clip the climatological data to the area 
* Must include
    * Path to file: path =
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max


In [ ]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,region_title=None,path=None):
    if path == None:
        clim = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded

Function builds a square (polygon) of the area to plot
* Must include
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

In [ ]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

Assign variables to the various datasets for plotting

In [ ]:
ds_GOM = hist_local_chl(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,region_title='Gulf of Maine')
ds_GB = hist_local_chl(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67,region_title='Georges Bank')
ds_MAB = hist_local_chl(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73,region_title='Middle Atlantic Bight')
clim_GOM = hist_clim_local(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,region_title='Gulf of Maine')
clim_GB = hist_clim_local(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67,region_title='Georges Bank')
clim_MAB = hist_clim_local(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73,region_title='Middle Atlantic Bight')
location_GOM = bound_local(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68)
location_GB = bound_local(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67)
location_MAB = bound_local(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73)

In [ ]:
local_chl = [ds_GOM,ds_GB,ds_MAB]
clim_med = [clim_GOM,clim_GB,clim_MAB]
clim_5 = [clim_GOM*1.05,clim_GB*1.05,clim_MAB*1.05]
clim_10 = [clim_GOM*1.1,clim_GB*1.1,clim_MAB*1.1]
clim_15 = [clim_GOM*1.15,clim_GB*1.15,clim_MAB*1.15]
clim_20 = [clim_GOM*1.2,clim_GB*1.2,clim_MAB*1.2]
clim_25 = [clim_GOM*1.25,clim_GB*1.25,clim_MAB*1.25]
clim_30 = [clim_GOM*1.3,clim_GB*1.3,clim_MAB*1.3]

In [ ]:
#Set up figure
fig=plt.figure(figsize=(12,14))
ax1 = plt.subplot(2,2,1,projection=crs.PlateCarree())
ax2 = plt.subplot(2,2,2)
ax3 = plt.subplot(2,2,3)
ax4 = plt.subplot(2,2,4)
# Location Map
ax1.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax1.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax1.add_geometries(bathym, facecolor='none', edgecolor='grey', crs=cartopy.crs.PlateCarree())
ax1.add_geometries(location_GOM, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Gulf of Maine')
ax1.add_geometries(location_GB, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Georges Bank')
ax1.add_geometries(location_MAB, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Middle Atlantic Bight')
ax1.set_extent([-77,-62,37,47])
gl=ax1.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False
ax1.set_title('Chlorophyll a Climatological Median and Histogram Locations', fontsize=14)

#Histograms
hist_axes = [ax2,ax3,ax4]
region_title = ["Gulf of Maine","Georges Bank",'Middle Atlantic Bight']
for i, data in enumerate(local_chl):
    ax = hist_axes[i]
    ax.hist(data,bins=100)
    ax.set_xlim(0,4)
    ax.axvline(clim_med[i][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i][0], color='purple', label='5% Threshold')
    ax.axvline(clim_10[i][0], color='yellow',label='10% Threshold')
    ax.axvline(clim_15[i][0], color='orange',label='15% Threshold')
    ax.axvline(clim_20[i][0], color='green',label='20% Threshold')
    ax.axvline(clim_25[i][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i][0], color='turquoise',label='30% Threshold')
    ax.legend()
    ax.set_title(region_title[i])
plt.tight_layout()
fig.suptitle("Chlorophyll a Concentrations",fontsize=20)

#### Determining the percentage of datapoints that lie above each threshold
Using the variables defined above for plotting, compare each value in the data set to that area's climatological median threshold.
* Threshold options:
    * clim_med = climatological median of the area (must use index of 0-2 in order of GOM, GB, MAB)
    * clim_5 = 5% above the climatological median (Must use indexing for all threshold values)
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2
* Dataset options:
    * ds_GOM = Gulf of Maine subset data
    * ds_GB = Georges Bank subset data
    * ds_MAB = Middle Atlantic Bight subset data


In [ ]:
def percent_above_thld(threshold,index,data):
    threshold=threshold[index].values
    total_above = int((data>threshold).sum())
    percent = (total_above/len(data))*100
    return percent

Testing 5%, 10%, and 15% for each region

In [ ]:
GOM_5 = percent_above_thld(clim_5,0,ds_GOM)
print("Gulf of Maine 5%: " + str(GOM_5))
GOM_10 = percent_above_thld(clim_10,0,ds_GOM)
print("Gulf of Maine 10%: " + str(GOM_10))
GOM_15 = percent_above_thld(clim_15,0,ds_GOM)
print("Gulf of Maine 15%: " + str(GOM_15))
GB_5 = percent_above_thld(clim_5,1,ds_GB)
print("Georges Bank 5%: " + str(GB_5))
GB_10 = percent_above_thld(clim_10,1,ds_GB)
print("Georges Bank 10%: " + str(GB_10))
GB_15 = percent_above_thld(clim_15,1,ds_GB)
print("Georges Bank 15%: " + str(GB_15))
MAB_5 = percent_above_thld(clim_5,2,ds_MAB)
print("Middle Atlantic Bight 5%: " + str(MAB_5))
MAB_10 = percent_above_thld(clim_10,2,ds_MAB)
print("Middle Atlantic Bight 10%: " + str(MAB_10))
MAB_15 = percent_above_thld(clim_15,2,ds_MAB)
print("Middle Atlantic Bight 15%: " + str(MAB_15))

#### Shapefile Analysis

Creates the shapefile for each region

In [ ]:
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)

Clip the climatology data to the shapefile

In [ ]:
clim = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
#clim_med = clim.CHL_median
clim.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
clim.rio.write_crs("epsg:4326", inplace=True)
clim_MAB_s = clim.rio.clip(MAB_south_loc.geometry, shapefile.crs, drop=True)
clim_MAB_s = clim_MAB_s.CHL_median.mean(dim=['lat','lon'])
clipped_clim = clim.rio.clip(MAB_north_loc.geometry, shapefile.crs, drop=True)
clim_MAB_n = clipped_clim.CHL_median.mean(dim=['lat','lon'])
clipped_clim = clim.rio.clip(GB_whole_loc.geometry, shapefile.crs, drop=True)
clim_GB = clipped_clim.CHL_median.mean(dim=['lat','lon'])
clipped_clim = clim.rio.clip(GOM_west_loc.geometry, shapefile.crs, drop=True)
clim_GOM_w = clipped_clim.CHL_median.mean(dim=['lat','lon'])
clipped_clim = clim.rio.clip(GOM_east_loc.geometry, shapefile.crs, drop=True)
clim_GOM_e = clipped_clim.CHL_median.mean(dim=['lat','lon'])

Clip D8 data to the shapefiles

In [ ]:
ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
ds.rio.write_crs("epsg:4326", inplace=True)
clipped_ds = ds.rio.clip(MAB_south_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_south = clipped_ds.CHL_median.mean(dim=['lat','lon'])
clipped_ds = ds.rio.clip(MAB_north_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_north = clipped_ds.CHL_median.mean(dim=['lat','lon'])
clipped_ds = ds.rio.clip(GB_whole_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GB_whole = clipped_ds.CHL_median.mean(dim=['lat','lon'])
clipped_ds = ds.rio.clip(GOM_west_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_west = clipped_ds.CHL_median.mean(dim=['lat','lon'])
clipped_ds = ds.rio.clip(GOM_east_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_east = clipped_ds.CHL_median.mean(dim=['lat','lon'])

In [ ]:
local_chl = [MAB_south,MAB_north,GB_whole,GOM_west,GOM_east]
clim_med = [clim_MAB_s,clim_MAB_n,clim_GB,clim_GOM_w,clim_GOM_e]
clim_5 = [clim_MAB_s*1.05,clim_MAB_n*1.05,clim_GB*1.05,clim_GOM_w*1.05,clim_GOM_e*1.05]
clim_10 = [clim_MAB_s*1.10,clim_MAB_n*1.10,clim_GB*1.10,clim_GOM_w*1.10,clim_GOM_e*1.10]
clim_15 = [clim_MAB_s*1.15,clim_MAB_n*1.15,clim_GB*1.15,clim_GOM_w*1.15,clim_GOM_e*1.15]
clim_20 = [clim_MAB_s*1.2,clim_MAB_n*1.2,clim_GB*1.2,clim_GOM_w*1.2,clim_GOM_e*1.2]
clim_25 = [clim_MAB_s*1.25,clim_MAB_n*1.25,clim_GB*1.25,clim_GOM_w*1.25,clim_GOM_e*1.25]
clim_30 = [clim_MAB_s*1.3,clim_MAB_n*1.3,clim_GB*1.3,clim_GOM_w*1.3,clim_GOM_e*1.3]

In [ ]:
fig=plt.figure(figsize=(10,12))
ax1 = plt.subplot(3,2,1,projection=crs.PlateCarree()) #Locations map
ax2 = plt.subplot(3,2,2) # MAB South
ax3 = plt.subplot(3,2,3) # MAB North
ax4 = plt.subplot(3,2,4) # GB
ax5 = plt.subplot(3,2,5) # GOM West
ax6 = plt.subplot(3,2,6) # GOM East
# Location Map
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
map_projection = cartopy.crs.PlateCarree()
ax1.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax1.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax1.add_geometries(bathym, facecolor='none', edgecolor='grey', crs=cartopy.crs.PlateCarree())
MAB_south_loc.boundary.plot(ax=ax1, color='green', linewidth=1.5, label="Middle Atlantic Bight South")
MAB_north_loc.boundary.plot(ax=ax1, color='blue', linewidth=1.5, label="Middle Atlantic Bight North")
GB_whole_loc.boundary.plot(ax=ax1, color='orange', linewidth=1.5, label="Georges Bank")
GOM_west_loc.boundary.plot(ax=ax1, color='red', linewidth=1.5, label="Gulf of Maine West")
GOM_east_loc.boundary.plot(ax=ax1, color='magenta', linewidth=1.5, label="Gulf of Maine East")
ax1.set_extent([-77,-62,37,47])
ax1.legend(fontsize=8,bbox_to_anchor=(1,1),loc='upper left')
gl=ax1.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False
ax1.set_title('Chlorophyll a Histogram Locations', fontsize=14)

#Histograms
hist_axes = [ax2,ax3,ax4,ax5,ax6]
region_title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank",'Gulf of Maine West',"Gulf of Maine East"]
for i, dataa in enumerate(local_chl):
    ax = hist_axes[i]
    ax.hist(dataa,bins=100)
    ax.set_xlim(0,4)
    ax.axvline(clim_med[i][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i][0], color='purple', label='5% Threshold')
    ax.axvline(clim_10[i][0], color='yellow',label='10% Threshold')
    ax.axvline(clim_15[i][0], color='orange',label='15% Threshold')
    ax.axvline(clim_20[i][0], color='magenta',label='20% Threshold')
    ax.axvline(clim_25[i][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i][0], color='turquoise',label='30% Threshold')
    ax.legend(fontsize=8)
    ax.set_title(region_title[i])
plt.tight_layout()
fig.suptitle("Chlorophyll a Concentrations",fontsize=20,y=1.05)

In [ ]:
clim.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
clim.rio.write_crs("epsg:4326", inplace=True)
clim_NES = clim.rio.clip(shapefile.geometry, shapefile.crs, drop=True)
clim_NES = clim_NES.CHL_median.mean(dim=['lat','lon'])
ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
ds.rio.write_crs("epsg:4326", inplace=True)
clipped_ds = ds.rio.clip(shapefile.geometry.apply(mapping), shapefile.crs, drop=True)
full_NES = clipped_ds.mean(dim=['lat','lon'])
clim_5_NES = clim_NES*1.05
clim_10_NES = clim_NES*1.1
clim_15_NES = clim_NES*1.15
clim_20_NES = clim_NES*1.2
clim_25_NES = clim_NES*1.25
clim_30_NES= clim_NES*1.3

In [ ]:
fig=plt.hist(full_NES.CHL_median,bins=75)
plt.axvline(clim_NES, color='red',label='Median') #Pulls first value in list and then the 1 value in that value
plt.axvline(clim_5_NES, color='purple', label='5% Threshold')
plt.axvline(clim_10_NES, color='yellow',label='10% Threshold')
plt.axvline(clim_15_NES, color='orange',label='15% Threshold')
plt.axvline(clim_20_NES, color='magenta',label='20% Threshold')
plt.axvline(clim_25_NES, color='pink',label='25% Threshold')
plt.axvline(clim_30_NES, color='turquoise',label='30% Threshold')
plt.legend(fontsize=8)
plt.title("NES Chlorophyll a")

#### Calculating the percentage of points above the threshold
Using the variables defined above for plotting and percent_above_thld(), compare each value in the data set to that area's climatological median threshold.
* Threshold options: (Must use index to get the correct area)
    * clim_med = climatological median of the area
    * clim_5 = 5% above the climatological median
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2, 3, 4 (in the order of: Middle Atlantic Bight South, Middle Atlantic Bight North, Georges Bank, Gulf of Maine West, Gulf of Maine East)
* Dataset options:
    * MAB_south = Middle Atlantic Bight South data
    * MAB_north = Middle Atlantic Bight North data
    * GB_whole = Georges Bank data
    * GOM_west = Gulf of Maine West data
    * GOM_east = Gulf of Maine East data

In [ ]:
MABS_5 = percent_above_thld(clim_5,0,MAB_south)
print("Middle Atlantic Bight South 5%: " + str(MABS_5))
MABS_10 = percent_above_thld(clim_10,0,MAB_south)
print("Middle Atlantic Bight South 10%: " + str(MABS_10))
MABS_15 = percent_above_thld(clim_15,0,MAB_south)
print("Middle Atlantic Bight South 15%: " + str(MABS_15))
MABN_5 = percent_above_thld(clim_5,1,MAB_north)
print("Middle Atlantic Bight North 5%: " + str(MABN_5))
MABN_10 = percent_above_thld(clim_10,1,MAB_north)
print("Middle Atlantic Bight North 10%: " + str(MABN_10))
MABN_15 = percent_above_thld(clim_15,1,MAB_north)
print("Middle Atlantic Bight North 15%: " + str(MABN_15))
GBW_5 = percent_above_thld(clim_5,2,GB_whole)
print("Georges Bank 5%: " + str(GBW_5))
GBW_10 = percent_above_thld(clim_10,2,GB_whole)
print("Georges Bank 10%: " + str(GBW_10))
GBW_15 = percent_above_thld(clim_15,2,GB_whole)
print("Georges Bank 15%: " + str(GBW_15))
GOMW_5 = percent_above_thld(clim_5,3,GOM_west)
print("Gulf of Maine West 5%: " + str(GOMW_5))
GOMW_10 = percent_above_thld(clim_10,3,GOM_west)
print("Gulf of Maine West 10%: " + str(GOMW_10))
GOMW_15 = percent_above_thld(clim_15,3,GOM_west)
print("Gulf of Maine West 15%: " + str(GOMW_15))
GOME_5 = percent_above_thld(clim_5,4,GOM_east)
print("Gulf of Maine East 5%: " + str(GOME_5))
GOME_10 = percent_above_thld(clim_10,4,GOM_east)
print("Gulf of Maine East 10%: " + str(GOME_10))
GOME_15 = percent_above_thld(clim_15,4,GOM_east)
print("Gulf of Maine East 15%: " + str(GOME_15))

#### Plotting the histograms centered on the median (Keep working on this)

In [ ]:
def percent_deviation(dataset,index,median):
    top = dataset.squeeze()-median[index][0]
    fraction = top/median[index][0]
    percent_dev = fraction*100
    return percent_dev

In [ ]:
MAB_south_per_dev = percent_deviation(MAB_south,0,clim_med)
MAB_north_per_dev = percent_deviation(MAB_north,1,clim_med)
GB_per_dev = percent_deviation(GB_whole,2,clim_med)
GOM_west_per_dev = percent_deviation(GOM_west,3,clim_med)
GOM_east_per_dev = percent_deviation(GOM_east,4,clim_med)
full_NES_per_dev = ((full_NES.squeeze()-clim_NES)/clim_NES)*100

Plotting the histograms

In [ ]:
custom_bins=[-10,-5,0,5,10,15,20,25,30,35]
fig,axes=plt.subplots(2,3,figsize=(25,14))
axes[0].hist(MAB_south_per_dev, bins=custom_bins, edgecolor='black', linewidth=1)
axes[0].set_title("Middle Atlantic Bight South")
axes[1].hist(MAB_north_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[1].set_title("Middle Atlantic Bight North")
axes[2].hist(GB_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[2].set_title("Georges Bank")
axes[3].hist(GOM_west_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[3].set_title("Gulf of Maine West")
axes[4].hist(GOM_east_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[4].set_title("Gulf of Maine East")
axes[5].hist(full_NES_per_dev,bins=custom_bins,edgecolor='black',linewidth=1)
axes[5].set_title("NES")

In [ ]:
#Adding variable to netCDF file
from netCDF4 import Dataset
path = r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NWA_4KM_DAY8\CHL\D8_19980102_19980109-OCCCI-CHL-NWA-STATS.nc'
ds = xr.open_dataset(path)
cdf = bloom_mask_numeric(path=path) #Creates masked variable
ds['five_percent_thld'] = cdf #Adds variable to ds
ds.to_netcdf(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NWA_4KM_DAY8\CHL\D8_19980102_19980109-OCCCI-CHL-NWA-STATS_with_threshold.nc') #Renames the file and saves it with the new variable

### Rate of Change

#### Step 0: Test with climatology data

In [ ]:
ds = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_2023*')

#### Spatially averaged rate of change

Thed rate_of_change function spatially averages the chlorophyll a data, smooths the data using a lowess smoother, and then find the rates of change between each data point, creating an array of rates of change.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)

In [ ]:
def rate_of_change(lat_min,lat_max,lon_min,lon_max,path=None):
    if path == None:
        files = get_prod_files('CHL',map_region='NES',period='WEEK')
        clim_med = xr.open_mfdataset(files)
    else:
        clim_med = xr.open_mfdataset(path)
    clim_med = clim_med.CHL_median
    clim_med = hist_local_chl(lat_min,lat_max,lon_min,lon_max,path)
    time = clim_med.time.astype('int64') #Changes time values to integers for smoothing
    clim_median=clim_med.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(clim_median['Chl_a'],time,frac=0.08)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    clim_roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return clim_roc


roc = rate_of_change(44,43,-69,-68,r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_2023*')
#print(roc)

The max_roc function builds on the rate_of_change function and finds the maximum rates of change in the array based on the parameters set in the function.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)
    * Distance = number of days between peaks (set to 10 days by default)
    * Prominence = the compared value of a rate of change to the next in order for it to be a peak (set to 0.02 by default)

In [ ]:
def max_roc(lat_min,lat_max,lon_min,lon_max,path,days=10,prm=0.02): #Days between peaks and the relative height of each peak value
    roc = rate_of_change(lat_min,lat_max,lon_min,lon_max,path)
    roc_max = find_peaks(roc,distance=days,prominence=prm)
    return roc_max

roc_max_values = max_roc(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,path=r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_2023*')
print(roc_max_values)

Using Lowess smoothing function

In [ ]:
data = hist_local_chl(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68) #Slices chl-a data into a 1 degree box (in Gulf of Maine)
fig=plt.plot(data.time.values,data, label="Raw Chl-a") #Plots the time series of raw data
time=data.time.astype('int64') # Transforms datetime into integers for smoothing (takes the dates in data and makes them integer values)
smoothed = sm.nonparametric.smoothers_lowess.lowess(data,time,frac=0.08) #smooths data (y-axis, x-axis, fraction)
x_smooth = smoothed[:,0] #Separates the data, grabbing the time values
y_smooth = smoothed[:,1] #Separates the data, grabbing the chlorophyll values
plt.plot(pd.to_datetime(x_smooth),y_smooth, label="Smoothed Chl-a")
plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
plt.xlabel("Date")
plt.title("Chlorophyll a in the Gulf of Maine in 2023")
plt.legend()

#### Step 1: Calculate instantaneous rate of change for each day

In [ ]:
def instant_rate_of_change(path=None):
    if path == None:
        #data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        data = xr.open_mfdataset(path)
    time = data.time.astype('int64') #Changes time values to integers for smoothing
    clim_median=data.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(clim_median['Chl_a'],time,frac=0.08)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    clim_roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return clim_roc


roc = instant_rate_of_change()
#print(roc)

#### Step 2: Set parameters for bloom intiation dates
1. Number of days between start dates
1. Prominence: height of slope peak compared to relative surrounding data
1. Height: Height required (min and/or max) for a peak

#### Step 3: Identify maximum rates of change
Use .find_peaks() function or .find_peaks_cwt() if the data is too noisy

#### Step 4: Extract dates of maximum rates of change

## Part 2: Quantify the Number of Phytoplankton Bloom Days Per Year

#### Actual number of bloom days per year over the time series

Based on the method(s) chosen in Part 1:
1. For each year: 365 - bloom
1. Plot the time series of the data to verify qualitatively

Quantifying bloom days
1. Separate the data out by year
    1. Run the mask for just yearly data?
1. How many daily files meet the bloom criteria?
1. Subtract the bloom daily files from the number of days that year.

Plotting the time series

#### Average number of bloom days over the time series

Using the data from above:
1. Calculate the mean (median) number of bloom days for the time series
1. Create an array of the actual days of year
1. Run median (and mean) statistics on the array

## Part 3: Quantify the Number of Phytoplankton Blooms Per Year

#### Actual number of phytoplankton blooms per year over the time series

1. Find the number of peaks (where derivative is 0) above the threshold/place where bloom conditions begin
2. Verify qualitatively with time series

#### Average number of phytoplankton blooms per year over the time series

## Part 4: Bloom Characteristics Analysis

#### Bloom start day

#### Duration of blooms

#### Maximum chlorophyll a

#### Minimum chlorophyll a

#### Mean chlorophyll a

#### Integrated chlorophyll a

#### Location of blooms
1. Find the center of gravity of major blooms

#### Periodicity of Blooms